In [ ]:
import cv2
import csv
import os
import torch
import numpy as np
from ultralytics import YOLO

# Liberar cualquier residuo en la memoria de la GPU
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# 1. CONFIGURACIÓN Y RUTAS DE ARCHIVOS
VIDEO_ENTRADA = 'C0142.MP4'
VIDEO_SALIDA = 'D:/resultado_C0142.mp4'
CSV_SALIDA = 'D:/detecciones_seguimiento.csv'

MODELO_GENERAL_PATH = 'yolo11n.pt'
MODELO_MATRICULAS_PATH = 'best.pt'

CLASES_INTERES = [0, 2, 3, 5, 7]
CLASES_VEHICULOS = [2, 3, 5, 7]
ANONIMIZAR = False

print("Cargando modelos YOLO...")
# device='cuda:0' para GPU, o 'cpu' si prefieres no usar la gráfica
DISPOSITIVO = 0 if torch.cuda.is_available() else 'cpu'

modelo_general = YOLO(MODELO_GENERAL_PATH)
modelo_matriculas = YOLO(MODELO_MATRICULAS_PATH)

cap = cv2.VideoCapture(VIDEO_ENTRADA)
if not cap.isOpened():
    raise FileNotFoundError(f"No se pudo abrir el vídeo: {VIDEO_ENTRADA}")

ancho = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
alto = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(VIDEO_SALIDA, fourcc, fps if fps > 0 else 25, (ancho, alto))

f_csv = open(CSV_SALIDA, mode='w', newline='', encoding='utf-8')
csv_writer = csv.writer(f_csv)
csv_writer.writerow([
    'fotograma', 'tipo_objeto', 'confianza', 
    'identificador_tracking', 'x1', 'y1', 'x2', 'y2', 'matricula'
])

conteo_unicos = {
    'person': set(),
    'car': set(),
    'motorcycle': set(),
    'bus': set(),
    'truck': set()
}

COLORES = {
    'person': (255, 105, 65),
    'car': (40, 200, 40),
    'motorcycle': (0, 165, 255),
    'bus': (200, 200, 0),
    'truck': (180, 0, 180),
    'matricula': (0, 255, 255)
}

print(f"Iniciando procesamiento ({total_frames} fotogramas) en dispositivo: {DISPOSITIVO}...")

frame_idx = 0

# torch.no_grad() evita calcular y retener grafos de gradientes en la VRAM
with torch.no_grad():
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_idx += 1
        
        # Inferencia con seguimiento (imgsz=640 para optimizar el uso de memoria)
        resultados_track = modelo_general.track(
            source=frame,
            persist=True,
            classes=CLASES_INTERES,
            tracker="botsort.yaml",
            imgsz=640,
            device=DISPOSITIVO,
            verbose=False
        )
        
        if resultados_track and len(resultados_track) > 0 and resultados_track[0].boxes is not None:
            cajas = resultados_track[0].boxes
            
            for box in cajas:
                x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                confianza = float(box.conf[0])
                clase_id = int(box.cls[0])
                nombre_clase = modelo_general.names[clase_id]
                track_id = int(box.id[0]) if box.id is not None else -1
                
                if track_id != -1 and nombre_clase in conteo_unicos:
                    conteo_unicos[nombre_clase].add(track_id)
                
                info_matricula = "No"
                caja_matricula = None
                
                if clase_id in CLASES_VEHICULOS:
                    vx1, vy1 = max(0, x1), max(0, y1)
                    vx2, vy2 = min(ancho, x2), min(alto, y2)
                    
                    if (vx2 - vx1) > 20 and (vy2 - vy1) > 20:
                        roi_vehiculo = frame[vy1:vy2, vx1:vx2]
                        res_placas = modelo_matriculas(roi_vehiculo, imgsz=320, conf=0.35, device=DISPOSITIVO, verbose=False)
                        
                        if res_placas and len(res_placas) > 0 and len(res_placas[0].boxes) > 0:
                            mejor_placa = res_placas[0].boxes[0]
                            px1_rel, py1_rel, px2_rel, py2_rel = map(int, mejor_placa.xyxy[0].tolist())
                            gx1, gy1 = vx1 + px1_rel, vy1 + py1_rel
                            gx2, gy2 = vx1 + px2_rel, vy1 + py2_rel
                            caja_matricula = (gx1, gy1, gx2, gy2)
                            info_matricula = f"[{gx1}:{gy1}:{gx2}:{gy2}]"
                
                csv_writer.writerow([
                    frame_idx, nombre_clase, f"{confianza:.2f}",
                    track_id, x1, y1, x2, y2, info_matricula
                ])
                
                color_obj = COLORES.get(nombre_clase, (0, 255, 0))
                cv2.rectangle(frame, (x1, y1), (x2, y2), color_obj, 2)
                cv2.putText(frame, f"ID:{track_id} {nombre_clase} {confianza:.2f}", 
                            (x1, max(20, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color_obj, 2)
                
                if caja_matricula is not None:
                    mx1, my1, mx2, my2 = caja_matricula
                    cv2.rectangle(frame, (mx1, my1), (mx2, my2), COLORES['matricula'], 2)
                    cv2.putText(frame, "Matricula", (mx1, max(15, my1 - 5)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.45, COLORES['matricula'], 2)

        # Panel de conteo
        cv2.rectangle(frame, (15, 15), (280, 160), (0, 0, 0), -1)
        cv2.rectangle(frame, (15, 15), (280, 160), (255, 255, 255), 1)
        cv2.putText(frame, f"Frame: {frame_idx}/{total_frames}", (25, 38), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)
        
        y_texto = 62
        for clase, ids in conteo_unicos.items():
            color_texto = COLORES.get(clase, (200, 200, 200))
            cv2.putText(frame, f"Total {clase}s: {len(ids)}", (25, y_texto), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.52, color_texto, 1)
            y_texto += 22
            
        out.write(frame)
        
        if frame_idx % 50 == 0 or frame_idx == total_frames:
            print(f"Progreso: {frame_idx}/{total_frames} fotogramas...")

cap.release()
out.release()
f_csv.close()

print("\n¡Procesamiento finalizado con éxito!")